# Effective Go — Practice Notebook

A hands-on tour of every major section in [Effective Go](https://go.dev/doc/effective_go).  
Each cell is self-contained: read the commentary, run the cell, then try the exercises.

**Setup:** requires [GoNB](https://github.com/janpfeifer/gonb) kernel  
```bash
go install github.com/janpfeifer/gonb@latest && \
  go install golang.org/x/tools/cmd/goimports@latest && \
  go install golang.org/x/tools/gopls@latest
  
gonb --install
```

## Prerequisites
### Install the libs if not available

In [ ]:
!go install golang.org/x/tools/cmd/goimports@latest
!go install golang.org/x/tools/gopls@latest

import (
	"fmt"
	"os"
)

%%

gopath := os.Getenv("GOPATH")
path := os.Getenv("PATH")
goroot := os.Getenv("GOROOT")
fmt.Println("GOROOT : ", goroot)
fmt.Println("GOPATH : ", gopath)
if strings.Contains(path, gopath+"/bin") {
	fmt.Println("GOPATH/bin is in PATH")
}else{
	fmt.Println("GOPATH/bin is NOT in PATH")
}



---
## 1. Formatting — `gofmt` conventions

Go delegates all formatting decisions to `gofmt`. Key rules:
- Tabs for indentation, not spaces
- No line length limit
- No parentheses on `if`, `for`, `switch`
- Opening brace on the **same** line as the control structure

In [ ]:
// Correct brace placement — opening brace same line
// If you put { on the next line, the lexer inserts a semicolon before it → compile error

import "fmt"

func formatDemo() {
	x := 10
	if x > 5 {          // brace here, not on next line
		fmt.Println("x is greater than 5")
	}

	// Struct field alignment — gofmt lines these up automatically
	type Server struct {
		host    string // short name
		port    int    // its port
		timeout int    // seconds
	}

	s := Server{host: "localhost", port: 8080, timeout: 30}
	fmt.Printf("%+v\n", s)
}
%%

formatDemo()

---
## 2. Commentary

- `//` line comments are the norm
- `/* */` for package-level doc blocks
- Doc comments appear **immediately before** top-level declarations (no blank line)
- `godoc` / `go doc` renders them

In [ ]:
// Package-level and function-level doc comments.
// A doc comment immediately before a declaration becomes its documentation.

import "fmt"

// Greet returns a greeting string for the given name.
// It never returns an empty string.
func Greet(name string) string {
	return "Hello, " + name + "!"
}

// MaxInt returns the larger of two integers.
func MaxInt(a, b int) int {
	if a > b {
		return a
	}
	return b
}
%%

fmt.Println(Greet("Gopher"))
fmt.Println(MaxInt(7, 3))

---
## 3. Names

### Rules from Effective Go
- Package names: short, lowercase, single word (`bufio`, not `buf_io`)
- Exported names start with uppercase — that is the visibility rule
- Getters: `Owner()` not `GetOwner()`; setters: `SetOwner()`
- One-method interfaces: name = method + `-er` → `Reader`, `Writer`, `Stringer`
- Use `MixedCaps` / `mixedCaps`, never underscores in identifiers

In [ ]:
import "fmt"

// Idiomatic naming: struct fields unexported, access via methods
type account struct {
	owner   string // unexported
	balance float64
}

// Owner is the getter — no "Get" prefix (Effective Go convention)
func (a *account) Owner() string { return a.owner }

// SetOwner is the setter
func (a *account) SetOwner(name string) { a.owner = name }

// Stringer interface — method must be called String(), not ToString()
func (a account) String() string {
	return fmt.Sprintf("Account{owner:%s balance:%.2f}", a.owner, a.balance)
}
%%

a := &account{owner: "Alice", balance: 1000.0}
fmt.Println(a)           // calls String() automatically
a.SetOwner("Bob")
fmt.Println(a.Owner())

---
## 4. Control Structures — `if`

Key idioms:
- Init statement inside `if` to scope a variable
- Avoid `else` after a `return` / `break` — let the happy path run down the page
- Error-guard pattern: handle the error early, keep success code unindented

In [ ]:
import (
	"errors"
	"fmt"
	"strconv"
)

// Init statement: err is scoped to the if block
func parseAndDouble(s string) (int, error) {
	if n, err := strconv.Atoi(s); err != nil {
		return 0, fmt.Errorf("parseAndDouble: %w", err)
	} else {
		return n * 2, nil
	}
}

// Idiomatic: no else — error returns early, success flows down
func divide(a, b float64) (float64, error) {
	if b == 0 {
		return 0, errors.New("division by zero")
	}
	return a / b, nil  // no else needed
}
%%

v, err := parseAndDouble("21")
fmt.Println(v, err)

_, err = parseAndDouble("abc")
fmt.Println(err)

r, _ := divide(10, 3)
fmt.Printf("10/3 = %.4f\n", r)

---
## 5. Control Structures — `for`

Go's `for` covers C's `for`, `while`, and infinite loop.  
Use `range` to iterate over slices, maps, strings, and channels.

In [ ]:
import "fmt"

%%

// Classic C-style for
sum := 0
for i := 0; i < 5; i++ {
	sum += i
}
fmt.Println("sum 0..4 =", sum)

// While-style (condition only)
n := 1
for n < 100 {
	n *= 2
}
fmt.Println("first power of 2 >= 100:", n)

// range over slice — index and value
langs := []string{"Go", "Rust", "Java"}
for i, lang := range langs {
	fmt.Printf("  %d: %s\n", i, lang)
}

// range over map
scores := map[string]int{"Alice": 95, "Bob": 87}
for name, score := range scores {
	fmt.Printf("  %s: %d\n", name, score)
}

// range over string — iterates Unicode code points
for pos, ch := range "Go🐹" {
	fmt.Printf("  byte %d: %c\n", pos, ch)
}

// Parallel assignment in for (reverse a slice)
a := []int{1, 2, 3, 4, 5}
for i, j := 0, len(a)-1; i < j; i, j = i+1, j-1 {
	a[i], a[j] = a[j], a[i]
}
fmt.Println("reversed:", a)


---
## 6. Control Structures — `switch`

- No `fallthrough` by default (opposite of C)
- Expression-less switch acts like `if-else-if`
- Cases can be comma-separated
- Labeled `break` to exit an enclosing loop from inside a switch

In [ ]:
import "fmt"

// Expression-less switch (acts like if-else-if)
func classify(n int) string {
	switch {
	case n < 0:
		return "negative"
	case n == 0:
		return "zero"
	case n < 10:
		return "small"
	default:
		return "large"
	}
}

// Comma-separated cases
func isVowel(c byte) bool {
	switch c {
	case 'a', 'e', 'i', 'o', 'u', 'A', 'E', 'I', 'O', 'U':
		return true
	}
	return false
}

%%

// Statements start here (everything below `%%` is wrapped in func main() by GoNB)
for _, v := range []int{-3, 0, 7, 42} {
	fmt.Printf("%3d → %s\n", v, classify(v))
}

fmt.Println()
for _, c := range "Hello" {
	fmt.Printf("  %c vowel=%v\n", c, isVowel(byte(c)))
}

// Labeled break: exits the outer for loop, not just the switch
words := []string{"foo", "bar", "STOP", "baz"}
fmt.Println()
Loop:
for _, w := range words {
	switch w {
	case "STOP":
		fmt.Println("breaking out of loop at STOP")
		break Loop  // exits the for, not just the switch
	default:
		fmt.Println("word:", w)
	}
}


---
## 7. Type Switch

Discovers the dynamic type of an `interface{}` variable at runtime.

In [ ]:
import "fmt"

func describe(i interface{}) string {
	switch v := i.(type) {
	case int:
		return fmt.Sprintf("int: %d", v)
	case string:
		return fmt.Sprintf("string: %q (len=%d)", v, len(v))
	case bool:
		return fmt.Sprintf("bool: %t", v)
	case []int:
		return fmt.Sprintf("[]int of len %d", len(v))
	default:
		return fmt.Sprintf("unknown type %T", v)
	}
}
%%

values := []interface{}{42, "hello", true, []int{1, 2, 3}, 3.14}
for _, v := range values {
	fmt.Println(describe(v))
}

---
## 8. Functions — Multiple Return Values

Go functions routinely return `(value, error)`. This eliminates in-band sentinel values like `-1` or `nil`.

In [ ]:
import (
	"errors"
	"fmt"
)

// Multiple returns: value + error
func safeSqrt(x float64) (float64, error) {
	if x < 0 {
		return 0, errors.New("cannot take sqrt of negative number")
	}
	// Newton's method
	z := x / 2
	for i := 0; i < 10; i++ {
		z -= (z*z - x) / (2 * z)
	}
	return z, nil
}

// Returns two values, both used
func minMax(nums []int) (min, max int) {
	min, max = nums[0], nums[0]
	for _, n := range nums[1:] {
		if n < min { min = n }
		if n > max { max = n }
	}
	return
}

%%

// Statements start here (everything below `%%` is wrapped in func main() by GoNB)
if v, err := safeSqrt(2.0); err != nil {
	fmt.Println("error:", err)
} else {
	fmt.Printf("sqrt(2) ≈ %.6f\n", v)
}

_, err := safeSqrt(-1)
fmt.Println(err)

lo, hi := minMax([]int{5, 2, 8, 1, 9, 3})
fmt.Printf("min=%d max=%d\n", lo, hi)


---
## 9. Named Return Values

Return values can be named. They are initialized to their zero values and act as documentation.  
A bare `return` returns their current values.

In [ ]:
import "fmt"

// Named return values — self-documenting and enables bare return
func split(sum int) (x, y int) {
	x = sum * 4 / 9
	y = sum - x
	return  // bare return — returns current x, y
}

// Named returns in a real utility: running stats
func stats(nums []float64) (mean, variance float64) {
	n := float64(len(nums))
	for _, v := range nums {
		mean += v
	}
	mean /= n
	for _, v := range nums {
		d := v - mean
		variance += d * d
	}
	variance /= n
	return  // returns mean and variance by name
}
%%

x, y := split(17)
fmt.Printf("split(17): x=%d y=%d\n", x, y)

data := []float64{2, 4, 4, 4, 5, 5, 7, 9}
m, v := stats(data)
fmt.Printf("mean=%.2f variance=%.2f\n", m, v)

---
## 10. Defer

- Schedules a call to run when the surrounding function returns
- Arguments are evaluated **when defer executes**, not when the call runs
- Multiple defers run in **LIFO** order
- Classic uses: `file.Close()`, `mu.Unlock()`, tracing

In [ ]:
import "fmt"

// LIFO order demo
func lifoDemo() {
	for i := 0; i < 5; i++ {
		defer fmt.Printf("%d ", i)  // arguments captured at defer time: 0,1,2,3,4
	}
}

// Tracing with defer — argument to un() is evaluated at defer-execute time
func trace(name string) string {
	fmt.Println("entering:", name)
	return name
}
func leave(name string) {
	fmt.Println("leaving:", name)
}

func outer() {
	defer leave(trace("outer"))  // trace("outer") runs NOW; leave() deferred
	fmt.Println("  inside outer")
	inner()
}
func inner() {
	defer leave(trace("inner"))
	fmt.Println("  inside inner")
}
%%

fmt.Print("LIFO output: ")
lifoDemo()
fmt.Println()
fmt.Println()
outer()

---
## 11. Data — `new` vs `make`

| | `new(T)` | `make(T, ...)` |
|---|---|---|
| What | Allocates zeroed memory | Initializes (not zeroes) |
| Returns | `*T` (pointer to zero value) | `T` (the value itself) |
| Works on | Any type | Only `slice`, `map`, `channel` |

In [ ]:
import "fmt"

type Point struct{ X, Y int }

%%

// new — zeroed, returns pointer
p := new(int)
fmt.Printf("new(int) → *int, value=%d, addr=%p\n", *p, p)

pt := new(Point)
pt.X, pt.Y = 3, 4
fmt.Println("new(Point):", *pt)

// make — initializes, returns the value (not a pointer)
// Slice: len=5, cap=10
s := make([]int, 5, 10)
fmt.Printf("make([]int,5,10): len=%d cap=%d %v\n", len(s), cap(s), s)

// Map: ready-to-use (unlike new which gives nil map)
m := make(map[string]int)
m["a"] = 1
fmt.Println("make(map):", m)

// Channel
ch := make(chan int, 2)
ch <- 10
ch <- 20
fmt.Println("buffered chan len:", len(ch))


---
## 12. Slices

Slices are a view into an underlying array: `(pointer, length, capacity)`.  
`append` grows them; slicing shares memory.

In [ ]:
import "fmt"

%%

// Slice is a window into an array
arr := [5]int{1, 2, 3, 4, 5}
s := arr[1:4]
fmt.Printf("arr=%v  s=%v len=%d cap=%d\n", arr, s, len(s), cap(s))

s[0] = 99  // modifies the underlying array
fmt.Println("after s[0]=99, arr:", arr)

// append — may allocate a new backing array
a := []int{1, 2, 3}
a = append(a, 4, 5)
b := []int{6, 7, 8}
a = append(a, b...)
fmt.Println("appended:", a)

// 2D slice — each row independently allocated
rows, cols := 3, 4
grid := make([][]int, rows)
for i := range grid {
	grid[i] = make([]int, cols)
	for j := range grid[i] {
		grid[i][j] = i*cols + j
	}
}
fmt.Println("grid:", grid)

// Copy avoids shared backing array
src := []int{10, 20, 30}
dst := make([]int, len(src))
copy(dst, src)
dst[0] = 999
fmt.Println("src:", src, "dst:", dst)


---
## 13. Maps

- Maps must be `make`-initialized before use (nil map panics on write)
- Comma-ok idiom to test key presence
- `delete` removes a key
- Iteration order is randomized

In [ ]:
import (
	"fmt"
	"sort"
)

// Map literal
func newWordCount() map[string]int {
	return map[string]int{
		"the": 5,
		"go":  3,
		"is":  2,
	}
}

%%

// Comma-ok: zero value vs absent key are indistinguishable without this
wordCount := newWordCount()
if count, ok := wordCount["go"]; ok {
	fmt.Println("go:", count)
}
if _, ok := wordCount["rust"]; !ok {
	fmt.Println("rust not in map")
}

// Increment pattern (safe: missing keys return 0)
words := []string{"apple", "banana", "apple", "cherry", "banana", "apple"}
freq := make(map[string]int)
for _, w := range words {
	freq[w]++
}

// Sorted output (map iteration order is random)
keys := make([]string, 0, len(freq))
for k := range freq {
	keys = append(keys, k)
}
sort.Strings(keys)
for _, k := range keys {
	fmt.Printf("  %s: %d\n", k, freq[k])
}

delete(freq, "apple")
fmt.Println("after delete(apple):", freq)


---
## 14. Interfaces

Interfaces are satisfied **implicitly** — no `implements` keyword.  
Any type with the right methods satisfies the interface.

In [ ]:
import (
	"fmt"
	"math"
)

// Interface — one method, named with -er suffix (Effective Go convention)
type Shaper interface {
	Area() float64
	Perimeter() float64
}

type Circle struct{ Radius float64 }
type Rectangle struct{ Width, Height float64 }

func (c Circle) Area() float64      { return math.Pi * c.Radius * c.Radius }
func (c Circle) Perimeter() float64 { return 2 * math.Pi * c.Radius }

func (r Rectangle) Area() float64      { return r.Width * r.Height }
func (r Rectangle) Perimeter() float64 { return 2 * (r.Width + r.Height) }

// printShape works on any Shaper — no explicit declaration needed
func printShape(s Shaper) {
	fmt.Printf("  area=%.3f  perimeter=%.3f\n", s.Area(), s.Perimeter())
}

// Interface composition
type Stringer interface{ String() string }
type StringShaper interface {
	Shaper
	Stringer
}

%%

shapes := []Shaper{
	Circle{Radius: 5},
	Rectangle{Width: 3, Height: 4},
}
for _, s := range shapes {
	fmt.Printf("%T\n", s)
	printShape(s)
}
fmt.Println("StringShaper composes Shaper + Stringer")


---
## 15. Goroutines

A goroutine is a lightweight thread managed by the Go runtime.  
`go f()` starts `f` concurrently. Goroutines share the same address space.

In [ ]:
import (
	"fmt"
	"sync"
	"time"
)

// Fan-out: launch many goroutines, collect results via WaitGroup
func worker(id int, wg *sync.WaitGroup) {
	defer wg.Done()
	time.Sleep(10 * time.Millisecond) // simulate work
	fmt.Printf("  worker %d done\n", id)
}

%%

var wg sync.WaitGroup
for i := 1; i <= 5; i++ {
	wg.Add(1)
	go worker(i, &wg)
}
wg.Wait()
fmt.Println("all workers done")

// Closure gotcha: capture loop variable correctly
var wg2 sync.WaitGroup
for i := 0; i < 3; i++ {
	wg2.Add(1)
	i := i  // shadow to capture current value
	go func() {
		defer wg2.Done()
		fmt.Printf("  goroutine sees i=%d\n", i)
	}()
}
wg2.Wait()


---
## 16. Channels

Channels are the primary communication mechanism between goroutines.  
> *"Do not communicate by sharing memory; share memory by communicating."*

In [ ]:
import "fmt"

// Unbuffered channel — sender blocks until receiver is ready
func producer(ch chan<- int, n int) {
	for i := 0; i < n; i++ {
		ch <- i * i
	}
	close(ch)
}
%%

ch := make(chan int)
go producer(ch, 6)
for v := range ch {  // range over channel reads until closed
	fmt.Print(v, " ")
}
fmt.Println()

// Buffered channel — send doesn't block until buffer full
buf := make(chan string, 3)
buf <- "a"
buf <- "b"
buf <- "c"
fmt.Println(<-buf, <-buf, <-buf)

// Pipeline: generate → square → print
gen := func(nums ...int) <-chan int {
	out := make(chan int)
	go func() {
		for _, n := range nums { out <- n }
		close(out)
	}()
	return out
}
sq := func(in <-chan int) <-chan int {
	out := make(chan int)
	go func() {
		for n := range in { out <- n * n }
		close(out)
	}()
	return out
}

for v := range sq(gen(2, 3, 4, 5)) {
	fmt.Print(v, " ")
}
fmt.Println()

---
## 17. `select`

`select` lets a goroutine wait on multiple channel operations simultaneously.  
It picks one ready case at random if multiple are ready.

In [ ]:
import (
	"fmt"
	"time"
)

// Timeout pattern using select + time.After
func slowOp(ch chan<- string) {
	time.Sleep(50 * time.Millisecond)
	ch <- "result"
}
%%

ch := make(chan string, 1)
go slowOp(ch)

select {
case res := <-ch:
	fmt.Println("got:", res)
case <-time.After(200 * time.Millisecond):
	fmt.Println("timeout")
}

// Non-blocking send using default
out := make(chan int, 1)
for _, v := range []int{1, 2, 3} {
	select {
	case out <- v:
		fmt.Println("sent:", v)
	default:
		fmt.Println("channel full, dropped:", v)
	}
}

---
## 18. Errors

Go treats errors as values. The idiomatic pattern is `(result, error)` returns.  
Use `fmt.Errorf("context: %w", err)` to wrap errors (Go 1.13+).

In [ ]:
import (
	"errors"
	"fmt"
)

// Sentinel error
var ErrNotFound = errors.New("not found")

// Custom error type — carries context
type ValidationError struct {
	Field   string
	Message string
}

func (e *ValidationError) Error() string {
	return fmt.Sprintf("validation error: field %q — %s", e.Field, e.Message)
}

func findUser(id int) (string, error) {
	users := map[int]string{1: "Alice", 2: "Bob"}
	if name, ok := users[id]; ok {
		return name, nil
	}
	return "", fmt.Errorf("findUser(%d): %w", id, ErrNotFound)
}

func createUser(name string, age int) error {
	if name == "" {
		return &ValidationError{Field: "name", Message: "must not be empty"}
	}
	if age < 0 || age > 150 {
		return &ValidationError{Field: "age", Message: "must be 0-150"}
	}
	return nil
}

// errors.Is — checks the chain for a sentinel
%%

_, err := findUser(99)
fmt.Println(err)
fmt.Println("is ErrNotFound:", errors.Is(err, ErrNotFound))

// errors.As — extracts a typed error from the chain
err = createUser("", 25)
var ve *ValidationError
if errors.As(err, &ve) {
	fmt.Printf("validation: field=%s msg=%s\n", ve.Field, ve.Message)
}

---
## 19. Panic and Recover

- `panic` unwinds the stack, running deferred functions
- `recover` (called inside a deferred function) catches the panic
- Use sparingly — prefer returning errors

In [ ]:
import "fmt"

// safeDiv recovers from a divide-by-zero panic
func safeDiv(a, b int) (result int, err error) {
	defer func() {
		if r := recover(); r != nil {
			err = fmt.Errorf("recovered panic: %v", r)
		}
	}()
	return a / b, nil  // panics if b == 0
}

// Panic with a string — conventional for programming errors
func mustPositive(n int) int {
	if n <= 0 {
		panic(fmt.Sprintf("mustPositive: got %d", n))
	}
	return n
}

func callMust(n int) (result int, err error) {
	defer func() {
		if r := recover(); r != nil {
			err = fmt.Errorf("%v", r)
		}
	}()
	return mustPositive(n), nil
}

%%

// All statements go below the %%
// Now we print the results sequentially instead of interweaving definitions

v, err := safeDiv(10, 2)
fmt.Println(v, err)

v, err = safeDiv(10, 0)
fmt.Println(v, err)

r, err := callMust(5)
fmt.Println(r, err)

r, err = callMust(-1)
fmt.Println(r, err)


---
## 20. Embedding

Go has no inheritance. Instead, types can **embed** other types, promoting their methods.  
This is composition, not subclassing.

In [ ]:
import "fmt"

type Animal struct{ Name string }
func (a Animal) Speak() string { return a.Name + " makes a sound" }

// Dog embeds Animal — promotes Animal's fields and methods
type Dog struct {
	Animal        // embedded, not a named field
	Breed  string
}

// Dog overrides Speak
func (d Dog) Speak() string { return d.Name + " says: Woof!" }

// Embedding interfaces — common for building composite interfaces
type Logger struct{ prefix string }
func (l Logger) Log(msg string) { fmt.Printf("[%s] %s\n", l.prefix, msg) }

type Service struct {
	Logger           // embed to get Log() promoted
	Name   string
}
%%

d := Dog{Animal: Animal{Name: "Rex"}, Breed: "Labrador"}
fmt.Println(d.Speak())      // Dog's own Speak
fmt.Println(d.Animal.Speak()) // explicitly call promoted method
fmt.Println(d.Name)           // promoted field

svc := Service{Logger: Logger{prefix: "INFO"}, Name: "auth"}
svc.Log("service started")  // promoted from Logger

---
## 21. The Blank Identifier `_`

Use `_` to discard values you don't need.  
Also used for side-effect imports and interface satisfaction checks.

In [ ]:
import "fmt"

// Interface satisfaction check at compile time
type Writer interface{ Write([]byte) (int, error) }
type MyWriter struct{}

func (w MyWriter) Write(b []byte) (int, error) { return len(b), nil }

var _ Writer = (*MyWriter)(nil) // compile-time interface check

%%

// Discard index in range
sum := 0
for _, v := range []int{1, 2, 3, 4, 5} {
	sum += v
}
fmt.Println("sum:", sum)

// Discard error when you're certain it can't fail (rare, use carefully)
nums := []int{3, 1, 4, 1, 5, 9, 2, 6}
maxVal := nums[0]
for _, n := range nums[1:] {
	if n > maxVal {
		maxVal = n
	}
}
fmt.Println("max:", maxVal)

fmt.Println("MyWriter satisfies Writer (compile-time checked)")


---
## 22. Concurrency — `sync.Mutex` and `sync/atomic`

When channels are overkill (e.g. protecting a counter), use a mutex.

In [ ]:
import (
	"fmt"
	"sync"
	"sync/atomic"
)

// Mutex-protected map
type SafeMap struct {
	mu sync.Mutex
	m  map[string]int
}

func NewSafeMap() *SafeMap { return &SafeMap{m: make(map[string]int)} }

func (sm *SafeMap) Inc(key string) {
	sm.mu.Lock()
	defer sm.mu.Unlock()  // defer ensures unlock even on panic
	sm.m[key]++
}

func (sm *SafeMap) Get(key string) int {
	sm.mu.Lock()
	defer sm.mu.Unlock()
	return sm.m[key]
}
%%

sm := NewSafeMap()
var wg sync.WaitGroup
for i := 0; i < 100; i++ {
	wg.Add(1)
	go func() {
		defer wg.Done()
		sm.Inc("hits")
	}()
}
wg.Wait()
fmt.Println("hits:", sm.Get("hits"))  // must be 100

// atomic counter — simpler when you only need increment/load
var counter int64
var wg2 sync.WaitGroup
for i := 0; i < 1000; i++ {
	wg2.Add(1)
	go func() {
		defer wg2.Done()
		atomic.AddInt64(&counter, 1)
	}()
}
wg2.Wait()
fmt.Println("atomic counter:", atomic.LoadInt64(&counter))

---
## 23. Putting It Together — A Mini CLI Tool

Combines: multiple returns, interfaces, goroutines, channels, defer, error handling.

In [ ]:
import (
	"fmt"
	"sort"
	"strings"
	"sync"
)

// Processor — a one-method interface (Effective Go naming: -er suffix)
type Processor interface {
	Process(input string) (string, error)
}

type UpperProcessor struct{}
func (u UpperProcessor) Process(s string) (string, error) {
	if s == "" { return "", fmt.Errorf("empty input") }
	return strings.ToUpper(s), nil
}

type WordCountProcessor struct{}
func (wc WordCountProcessor) Process(s string) (string, error) {
	words := strings.Fields(s)
	counts := make(map[string]int)
	for _, w := range words { counts[w]++ }
	keys := make([]string, 0, len(counts))
	for k := range counts { keys = append(keys, k) }
	sort.Strings(keys)
	var sb strings.Builder
	for _, k := range keys {
		fmt.Fprintf(&sb, "%s:%d ", k, counts[k])
	}
	return strings.TrimSpace(sb.String()), nil
}

type Result struct {
	Name   string
	Output string
	Err    error
}

func runPipeline(input string, procs map[string]Processor) []Result {
	results := make([]Result, 0, len(procs))
	var (
		mu sync.Mutex
		wg sync.WaitGroup
	)
	for name, p := range procs {
		wg.Add(1)
		name, p := name, p
		go func() {
			defer wg.Done()
			out, err := p.Process(input)
			mu.Lock()
			results = append(results, Result{Name: name, Output: out, Err: err})
			mu.Unlock()
		}()
	}
	wg.Wait()
	sort.Slice(results, func(i, j int) bool { return results[i].Name < results[j].Name })
	return results
}

%%

procs := map[string]Processor{
	"upper": UpperProcessor{},
	"words": WordCountProcessor{},
}
for _, r := range runPipeline("the quick brown fox jumps over the lazy fox", procs) {
	if r.Err != nil {
		fmt.Printf("[%s] error: %v\n", r.Name, r.Err)
	} else {
		fmt.Printf("[%s] %s\n", r.Name, r.Output)
	}
}


---
## 12. Context Package

The `context` package provides a way to pass request-scoped values, cancelation signals, and deadlines across API boundaries to all the goroutines involved in handling a request.
- `context.Background()`: Empty context, usually the root.
- `context.WithCancel()`: Returns a copy of parent with a new Done channel.
- `context.WithTimeout()`: Cancels context automatically after timeout.
- `context.WithValue()`: Attaches key-value pairs.

In [ ]:
import (
	"context"
	"fmt"
	"time"
)

func doWork(ctx context.Context, duration time.Duration) error {
	select {
	case <-time.After(duration):
		fmt.Println("Work finished")
		return nil
	case <-ctx.Done():
		fmt.Println("Work cancelled:", ctx.Err())
		return ctx.Err()
	}
}
%%
ctx, cancel := context.WithTimeout(context.Background(), 1*time.Second)
defer cancel()

fmt.Println("Starting work...")
err := doWork(ctx, 2*time.Second) // Takes longer than timeout
if err != nil {
	fmt.Println("Error:", err)
}

---
## 13. Sync Package

The `sync` package provides basic synchronization primitives such as mutual exclusion locks.
- `sync.Mutex`: A mutual exclusion lock.
- `sync.RWMutex`: A reader/writer mutual exclusion lock.
- `sync.WaitGroup`: Waits for a collection of goroutines to finish.

In [ ]:
import (
	"fmt"
	"sync"
)
%%
var wg sync.WaitGroup
var mu sync.Mutex
var counter int

fmt.Println("Starting goroutines...")
for i := 0; i < 5; i++ {
	wg.Add(1)
	go func(id int) {
		defer wg.Done()
		mu.Lock()
		counter++
		fmt.Printf("Goroutine %d incremented counter to %d\n", id, counter)
		mu.Unlock()
	}(i)
}

wg.Wait()
fmt.Println("Final counter value:", counter)

---
## 14. Testing Package

The `testing` package provides support for automated testing of Go packages.
Since this is a notebook, we typically test code in standard project files, but we can document the pattern here or use the `testing` package programmatically.
A unit test function takes a pointer to `testing.T` and uses it to report failures.

In [ ]:
import (
	"fmt"
	"testing"
)

// A simple function to test
func Abs(x int) int {
	if x < 0 {
		return -x
	}
	return x
}

// A standard test function (normally in a _test.go file)
func TestAbs(t *testing.T) {
	got := Abs(-1)
	if got != 1 {
		t.Errorf("Abs(-1) = %d; want 1", got)
	}
}
%%
// In GoNB, tests are typically run from external files or using special macros, 
// but here we manually invoke the test logic for demonstration.
t := &testing.T{}
TestAbs(t)
if t.Failed() {
	fmt.Println("Test failed!")
} else {
	fmt.Println("Test passed! (Abs function works correctly)")
}

---
## 15. HTTP Package

The `net/http` package provides HTTP client and server implementations.
- `http.Get`, `http.Post` for client requests.
- `http.HandleFunc`, `http.ListenAndServe` for server.

In [ ]:
import (
	"fmt"
	"io"
	"net/http"
)
%%
// Simple HTTP GET request
resp, err := http.Get("https://httpbin.org/get")
if err != nil {
	fmt.Println("Error:", err)
} else {
	defer resp.Body.Close()
	body, _ := io.ReadAll(resp.Body)
	fmt.Printf("Status: %s\n", resp.Status)
	fmt.Printf("Body snippet: %s\n", string(body)[:70])
}

---
## 16. Full Implementation: HTTP Server with Logger, Context, and DB

Putting it all together:
- Setting up an HTTP Server with routing.
- Injecting a logger and wrapping handlers.
- Using Context for timeout/cancelation logic.
- Mocking database calls.

In [ ]:
import (
	"context"
	"fmt"
	"log"
	"net/http"
	"net/http/httptest"
	"time"
)

// Mock DB call
func fetchUserFromDB(ctx context.Context, id string) (string, error) {
	// Simulate DB query delay
	select {
	case <-time.After(50 * time.Millisecond):
		return fmt.Sprintf("{ \"id\": \"%s\", \"name\": \"Alice\" }", id), nil
	case <-ctx.Done():
		return "", ctx.Err() // Propagate cancellation or timeout
	}
}

// Handler with Context and Logger
func userHandler(w http.ResponseWriter, r *http.Request) {
	ctx := r.Context()
	id := r.URL.Query().Get("id")
	
	if id == "" {
		http.Error(w, "Missing 'id' parameter", http.StatusBadRequest)
		return
	}

	log.Printf("Handling request for user %s", id)
	
	// Call DB with the request context
	data, err := fetchUserFromDB(ctx, id)
	if err != nil {
		http.Error(w, "DB error: "+err.Error(), http.StatusInternalServerError)
		return
	}
	
	w.Header().Set("Content-Type", "application/json")
	fmt.Fprintf(w, data)
}
%%
// Instead of http.ListenAndServe, we use httptest to safely run and test the server in this notebook.

// Setup server and handler
mux := http.NewServeMux()
mux.HandleFunc("/user", userHandler)
server := httptest.NewServer(mux)
defer server.Close() // Cleanup on exit

fmt.Println("Test Server started at", server.URL)

// 1. Successful request
resp, err := http.Get(server.URL + "/user?id=123")
if err == nil {
	defer resp.Body.Close()
	
	var body []byte
	buf := make([]byte, 1024)
	n, _ := resp.Body.Read(buf)
	body = buf[:n]
	fmt.Println("HTTP 200 Response:", string(body))
}

// 2. Request with simulated timeout
// We'll create a client with a short timeout to see context cancellation propagating
client := &http.Client{
	Timeout: 10 * time.Millisecond,
}
_, err = client.Get(server.URL + "/user?id=456")
if err != nil {
	fmt.Println("HTTP Client Timeout Error expected:", err)
}

---
## 17. Fan-In Pattern (Merge Multiple Channels into One)

Key ideas:
- Multiple input channels feeding into a single output channel.
- Use `select` to multiplex inputs.
- Handle channel closure correctly (`ok` pattern).
- Disable closed channels using `nil` to avoid infinite zero values.
- Ensure output channel is closed exactly once after all inputs are drained.
- Avoid goroutine leaks and blocking issues.

Gotchas covered in code:
- Closed channel returns zero value forever if `ok` not checked.
- `select` will keep picking closed channels unless disabled.
- `time.After` inside loop can leak timers.
- Sender should close channels, not receiver.
- Output channel must be closed after all inputs are done.


In [ ]:
package main

import (
	"fmt"
	"time"
)

func main() {
	ch1 := make(chan int)
	ch2 := make(chan int)
	out := make(chan int)

	// producer 1
	go func() {
		defer close(ch1) // ✅ sender closes channel
		for i := 0; i < 5; i++ { // ⚠️ range 5 is invalid
			ch1 <- i
		}
	}()

	// producer 2
	go func() {
		defer close(ch2)
		for i := 5; i < 10; i++ {
			ch2 <- i
		}
	}()

	// fan-in (merge ch1 + ch2 into out)
	go func() {
		defer close(out) // ⚠️ must close ONLY after all inputs are drained

		for ch1 != nil || ch2 != nil {
			select {
			case v, ok := <-ch1:
				if !ok {
					ch1 = nil // ⚠️ disable closed channel (otherwise infinite zero values)
					continue
				}
				out <- v // ⚠️ can block if no consumer (backpressure)

			case v, ok := <-ch2:
				if !ok {
					ch2 = nil // ⚠️ required to avoid select picking closed channel forever
					continue
				}
				out <- v
			}
		}
	}()

	// consumer (runs in main goroutine)
	for v := range out { // ⚠️ blocks until values arrive
		fmt.Println(v)
	}

	// ⚠️ NOTE: if out is never closed → this loop blocks forever

	// ⚠️ BAD PATTERN (not used here):
	// for {
	//     select {
	//     case <-time.After(1 * time.Second): // ❌ creates new timer each loop → leak
	//     }
	// }

	_ = time.Second // avoid unused import in notebook
}


---
## 18. Fan-Out Pattern (Distribute Work Across Workers)

Key ideas:
- Multiple workers consume from the same input channel.
- Work is automatically distributed among workers.
- Use `sync.WaitGroup` to track worker completion.
- Output channel is shared across workers (fan-in happens implicitly).
- Close output channel ONLY after all workers finish.

Gotchas covered in code:
- Forgetting `go` → no concurrency (runs sequentially).
- Not closing input → workers block forever.
- Closing output too early → panic (send on closed channel).
- Not using WaitGroup → cannot safely close output.
- Unbuffered channels can create backpressure.
- Loop variable capture (if using closures incorrectly).


In [ ]:
package main

import (
	"fmt"
	"sync"
)

func main() {
	jobs := make(chan int)
	results := make(chan int)

	// producer
	go func() {
		defer close(jobs) // ✅ sender closes input channel
		for i := 0; i < 10; i++ { // ⚠️ range 10 is invalid
			jobs <- i
		}
	}()

	// worker function (fan-out)
	worker := func(id int, in <-chan int, out chan<- int, wg *sync.WaitGroup) {
		defer wg.Done()

		for job := range in { // ⚠️ exits automatically when jobs is closed
			result := job * job
			out <- result // ⚠️ can block if consumer is slow (backpressure)
		}
	}

	var wg sync.WaitGroup
	numWorkers := 3

	// start workers
	for i := 0; i < numWorkers; i++ {
		wg.Add(1)
		go worker(i, jobs, results, &wg) // ⚠️ missing 'go' → runs sequentially (common bug)
	}

	// close results after all workers finish
	go func() {
		wg.Wait() // ✅ ensures all workers are done
		close(results) // ⚠️ must close ONLY once, after all sends complete
	}()

	// consumer (main goroutine)
	for res := range results { // ⚠️ blocks until results arrive
		fmt.Println(res)
	}

	// ⚠️ If results is never closed → infinite block here

	// ⚠️ BAD PATTERN (not used here):
	// capturing loop variable incorrectly
	// for i := 0; i < numWorkers; i++ {
	//     go func() {
	//         fmt.Println(i) // ❌ all goroutines print same value
	//     }()
	// }
}
